# 03 - Delayed-reaction event study

**What:** measure immediate vs delayed reaction around each flow event and rank by the delayed-flow opportunity score.

**Why:** we only want events where the immediate move is small but the latent forced demand (vs ADV) is large and the effective date is still ahead.

**Key metrics:** `immediate_reaction` (ann close -> next open), `delayed_return` (next open -> effective), `underreaction_score`, and `delayed_flow_opportunity_score`.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from index_flow.config import load_config
from index_flow.event_builder import build_and_enrich
cfg = load_config(); cfg.ensure_dirs()
events, feats = build_and_enrich(cfg, use_fundamentals=True)
print('events:', len(events))

In [ ]:
if events.empty:
    print('No events to study yet.')
else:
    cols = ['asx_ticker','event_type','provider','flow_pressure','immediate_reaction','delayed_return','underreaction_score','delayed_flow_opportunity_score']
    top = events.sort_values('delayed_flow_opportunity_score', ascending=False)
    display(top[[c for c in cols if c in top.columns]].head(20))

In [ ]:
# Average cumulative drift: immediate vs delayed windows
if not events.empty:
    wcols = [c for c in events.columns if c.startswith('es_')]
    means = events[wcols].apply(pd.to_numeric, errors='coerce').mean()
    ax = means.plot(kind='bar', figsize=(10,4), title='Mean return by event-study window')
    ax.axhline(0, color='k', lw=0.8); plt.tight_layout(); plt.show()